# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
This reduced multicollinearity and improved balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.

In [17]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

# ----------------------------
# Experiment selection
# ----------------------------
exp_name = "E8_clean_vascular"
H = 5  # horizon years used in build_cohort notebook

OUT_DIR = os.path.join("..", "output")

# Build the cohort filename dynamically
cohort_file = f"cohort_{exp_name}_H{H}.csv"
DATA_PATH = os.path.join(OUT_DIR, cohort_file)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find cohort file: {DATA_PATH}\n"
        f"Make sure 01_build_cohort exported: {cohort_file}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", cohort_file, df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Loaded: cohort_E8_clean_vascular_H5.csv (38341, 42)
label
Censored         23300
NoDementia_Hy    11083
AD                3068
MixedOrOther       421
PD_LBD             297
Vascular           172
Name: count, dtype: int64


,NACCID,baseline_date,label,max_follow_years,has_H_followup,has_dementia_H,NACCAGE,SEX,RACE,HISPANIC,...,RIGIDL,RIGIDR,SLOWINGL,SLOWINGR,PARKGAIT,GAITDIS,VASC_RISK_COUNT,STROKE_TIA_ANY,VASC_MED_COUNT,GAIT_ANY
0,NACC000011,2006-04-17,Censored,3.296372,False,False,62.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0,0.0,0
1,NACC000034,2015-07-16,NoDementia_Hy,6.986995,True,False,79.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,1.0,0,2.0,0
2,NACC000067,2012-05-23,Censored,2.088980,False,False,60.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0,1.0,1
3,NACC000073,2016-05-20,Censored,0.000000,False,False,44.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0,0.0,0
4,NACC000138,2023-07-13,Censored,1.043121,False,False,54.0,2.0,50.0,1.0,...,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0,0.0,0


In [18]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figs_baseline")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [19]:
def fix_categorical_codes(X):
    X = X.copy()
    CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X.columns]
    STD_MISS = {-4, 9}

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(STD_MISS)), np.nan)
        X[c] = s

    if "RACE" in X.columns:
        X["RACE"] = X["RACE"].mask(X["RACE"].isin([99]), np.nan)

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        X[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

    # Optional subgroup bin
    if "EDUC" in X.columns:
        educ_num = pd.to_numeric(X["EDUC"], errors="coerce")
        X["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

    X = X.replace({pd.NA: np.nan})
    for c in X.columns:
        if X[c].dtype == "bool":
            X[c] = X[c].astype(int)
    return X

In [20]:
import numpy as np
import pandas as pd


df4 = df.copy()  # if your df is already the cohort dataframe; otherwise remove this line

# Keep only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_Hy", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Drop IDs/labels
#y = df4["label"].astype(str)
# drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
#drop_cols = [c for c in [ "NACCID", "baseline_date", "label", "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


def prepare_experiment_df(df, exp_name):
    df = df.copy()

    # columns always dropped
    drop_cols = [c for c in ["NACCID","baseline_date","label","has_H_followup","has_dementia_H","max_follow_years","abs_days_from_baseline"]
                 if c in df.columns]

    # ---- E8: engineered vascular burden; DROP raw components to avoid collinearity ----
    if exp_name == "E8_clean_vascular":
        raw_vasc = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
        drop_cols += [c for c in raw_vasc if c in df.columns]

        raw_meds = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
        drop_cols += [c for c in raw_meds if c in df.columns]

        # keep GAITDIS if you want; GAIT_ANY is enough typically:
        # drop_cols += [c for c in ["GAITDIS"] if c in df.columns]

    # ---- E9: exec ratio; DROP raw Trails ----
    if exp_name == "E9_clean_exec_ratio":
        drop_cols += [c for c in ["TRAILA","TRAILB","TRAILA_missing","TRAILB_missing"] if c in df.columns]

    # ---- E10: build exec-memory axis (train-only z-scoring later), then DROP raw ----
    # Here we only mark what to drop AFTER we compute engineered features per split.
    # We'll handle this inside the training loop because we need train means/stds.
    # (So nothing special here yet.)

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["label"].astype(str)

    return X, y


from sklearn.model_selection import train_test_split

X, y = prepare_experiment_df(df4, exp_name)

# Apply categorical cleanup BEFORE splitting (safe because it does not use y)
X = fix_categorical_codes(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())
print("Test label counts:\n", y_test.value_counts())



def z_from_train(train_s, s):
    mu = train_s.mean()
    sd = train_s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s*0.0
    return (s - mu) / sd

if exp_name == "E10_exec_mem_axis_raw":
    needed = ["NACCMMSE", "NACCMOCA", "TRAILB"]
    missing = [c for c in needed if c not in X_train.columns]
    if missing:
        raise ValueError(f"E10 missing required columns: {missing}")

    # Z-scores using TRAIN stats only
    X_train["Z_TRAILB"] = z_from_train(X_train["TRAILB"], X_train["TRAILB"])
    X_test["Z_TRAILB"]  = z_from_train(X_train["TRAILB"], X_test["TRAILB"])

    mem_cols = [c for c in ["NACCMMSE","NACCMOCA"] if c in X_train.columns]
    Zm_train = []
    Zm_test  = []
    for c in mem_cols:
        Zm_train.append(z_from_train(X_train[c], X_train[c]))
        Zm_test.append(z_from_train(X_train[c], X_test[c]))

    X_train["Z_MEM"] = np.nanmean(np.vstack([z.values for z in Zm_train]), axis=0)
    X_test["Z_MEM"]  = np.nanmean(np.vstack([z.values for z in Zm_test]), axis=0)

    # Executive minus memory axis
    X_train["EXEC_MINUS_MEM"] = X_train["Z_TRAILB"] - X_train["Z_MEM"]
    X_test["EXEC_MINUS_MEM"]  = X_test["Z_TRAILB"]  - X_test["Z_MEM"]

    # Drop raw components to avoid collinearity
    drop_raw = [
        "TRAILA","TRAILB","NACCMMSE","NACCMOCA",
        "TRAILA_missing","TRAILB_missing","NACCMMSE_missing","NACCMOCA_missing",
        "Z_TRAILB","Z_MEM"
    ]
    X_train = X_train.drop(columns=[c for c in drop_raw if c in X_train.columns])
    X_test  = X_test.drop(columns=[c for c in drop_raw if c in X_test.columns])

    print("E10 engineered EXEC_MINUS_MEM added. Shapes now:", X_train.shape, X_test.shape)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

4-class rows: 14620
label
NoDementia_Hy    11083
AD                3068
PD_LBD             297
Vascular           172
Name: count, dtype: int64
Train: (11696, 37) Test: (2924, 37)
Train label counts:
 label
NoDementia_Hy    8866
AD               2454
PD_LBD            238
Vascular          138
Name: count, dtype: int64
Test label counts:
 label
NoDementia_Hy    2217
AD                614
PD_LBD             59
Vascular           34
Name: count, dtype: int64


In [21]:
csf_cols = ["CSFABETA", "CSFPTAU", "CSFTTAU"]
present_csf_cols = [c for c in csf_cols if c in df.columns]

if len(present_csf_cols) == len(csf_cols):
    df_csf = df[df[present_csf_cols].notna().all(axis=1)].copy()
    print("CSF subset size:", len(df_csf))
    print(df_csf["label"].value_counts())
else:
    print("CSF columns not present in this dataset:", 
          [c for c in csf_cols if c not in df.columns])

CSF columns not present in this dataset: ['CSFABETA', 'CSFPTAU', 'CSFTTAU']


## Split + preprocess + train

Train/test split

In [22]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
# )




Preprocessing pipelines (sparse for LR/boosting)

In [23]:
# cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
# num_cols = [c for c in X_train.columns if c not in cat_cols]

import pandas as pd

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))
print(sorted([c for c in X_train.columns if "VASC" in c or "EXEC" in c or "TRAIL" in c or "MMSE" in c or "MOCA" in c]))
print(sorted([c for c in X_train.columns]))



Num cols: 32 Cat cols: 5
['NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC_MED_COUNT', 'VASC_RISK_COUNT']
['ANIMALS', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'COGFLUC', 'EDUC', 'EDUC_BIN', 'GAITDIS', 'GAIT_ANY', 'HISPANIC', 'NACCAGE', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCUDSD', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SLOWINGL', 'SLOWINGR', 'STROKE_TIA_ANY', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC_MED_COUNT', 'VASC_RISK_COUNT']


In [24]:
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.over_sampling import SMOTE

## The block below has the SMOTE results while the block afterwards is the normal, however the normal has some better results.

In [25]:
# from sklearn.linear_model import LogisticRegression

# custom_w = {
#     "NoDementia_Hy": 1.0,
#     "AD": 2.0,
#     "PD_LBD": 6.0,
#     "Vascular": 8.0
# }

# smote = SMOTE(sampling_strategy="not majority", k_neighbors=2, random_state=42)

# clf_lr = ImbPipeline(steps=[
#     ("preprocess", preprocess),
#     ("smote", smote),
#     ("model", LogisticRegression(max_iter=8000, class_weight=None, C=0.1))
# ])

# clf_lr.fit(X_train, y_train)
# pred_lr = clf_lr.predict(X_test)

# print("=== Logistic Regression (SMOTE, C=0.1) ===")
# print(classification_report(y_test, pred_lr, digits=4))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
# print(confusion_matrix(y_test, pred_lr, labels=labels))


Model A: Logistic Regression (final settings)

In [26]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline
# from sklearn.base import clone
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
# import numpy as np

# # -----------------------------
# # 1) Systematic class weights (balanced) + optional vascular boost
# # -----------------------------
# classes = np.unique(y_train)

# base_w = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=y_train
# )
# class_weight_dict = dict(zip(classes, base_w))

# VAS_MULTIPLIER = 1.1   # try: 1.0, 1.2, 1.3, 1.5
# class_weight_dict["Vascular"] = class_weight_dict.get("Vascular", 1.0) * VAS_MULTIPLIER

# print("Final class weights:", class_weight_dict)

# # -----------------------------
# # 2) Fit multinomial logistic regression
# # -----------------------------
# clf_lr = Pipeline(steps=[
#     ("preprocess", clone(preprocess)),
#     ("model", LogisticRegression(
#         solver="lbfgs",          # supports multinomial automatically
#         max_iter=8000,
#         class_weight=class_weight_dict,
#         C=0.1
#     ))
# ])

# clf_lr.fit(X_train, y_train)

# # -----------------------------
# # 3) Default prediction
# # -----------------------------
# pred_lr = clf_lr.predict(X_test)

# # -----------------------------
# # 4) OPTIONAL: vascular threshold override (often boosts vascular recall)
# # -----------------------------
# USE_VASC_THRESHOLD = False
# T_VAS = 0.18  # try: 0.15–0.25

# if USE_VASC_THRESHOLD:
#     probs = clf_lr.predict_proba(X_test)
#     order = clf_lr.named_steps["model"].classes_
#     vas_idx = list(order).index("Vascular")
#     p_vas = probs[:, vas_idx]

#     pred_lr = pred_lr.copy()
#     pred_lr[p_vas >= T_VAS] = "Vascular"

# print("\n=== Logistic Regression (balanced weights + vascular boost + optional threshold) ===")
# print(classification_report(y_test, pred_lr, digits=4, labels=labels))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
# print(confusion_matrix(y_test, pred_lr, labels=labels))

In [27]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.6380    0.7980    0.7091       614
NoDementia_Hy     0.9447    0.8791    0.9107      2217
       PD_LBD     0.5000    0.6102    0.5496        59
     Vascular     0.3810    0.2353    0.2909        34

     accuracy                         0.8492      2924
    macro avg     0.6159    0.6307    0.6151      2924
 weighted avg     0.8648    0.8492    0.8539      2924

Balanced accuracy: 0.6306562835490068
[[ 490  101   16    7]
 [ 244 1949   19    5]
 [  16    6   36    1]
 [  18    7    1    8]]


Tuned Logistic Regression

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

# -----------------------------
# 1) Create train/validation split from TRAIN only (no test leakage)
# -----------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# -----------------------------
# 2) Fit LR on inner-train
# -----------------------------
clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_tr, y_tr)

# -----------------------------
# 3) Tune vascular threshold on validation set
# -----------------------------
proba_val = clf_lr.predict_proba(X_val)
class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

thresholds = np.linspace(0.05, 0.50, 20)
results = []

for T in thresholds:
    preds = []
    for p in proba_val:
        if p[vas_idx] >= T:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    ba = balanced_accuracy_score(y_val, preds)

    # vascular recall (manual)
    yv = np.array(y_val)
    pv = np.array(preds)
    denom = np.sum(yv == "Vascular")
    vrec = (np.sum((yv == "Vascular") & (pv == "Vascular")) / denom) if denom else 0.0

    results.append((T, ba, vrec))

best_T, best_ba, best_vrec = max(results, key=lambda x: x[1])
print(f"Best T_VAS (by val BA): {best_T:.3f} | val BA={best_ba:.4f} | val VascRecall={best_vrec:.4f}")

# -----------------------------
# 4) Retrain LR on FULL TRAIN and evaluate once on TEST using tuned threshold
# -----------------------------
clf_lr.fit(X_train, y_train)

proba_test = clf_lr.predict_proba(X_test)
preds_test = []

for p in proba_test:
    if p[vas_idx] >= best_T:
        preds_test.append("Vascular")
    else:
        preds_test.append(class_order[np.argmax(p)])

print("\n=== Logistic Regression (Vascular threshold tuned) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, preds_test))
print(confusion_matrix(y_test, preds_test, labels=labels))

Best T_VAS (by val BA): 0.145 | val BA=0.6361 | val VascRecall=0.4444

=== Logistic Regression (Vascular threshold tuned) ===
               precision    recall  f1-score   support

           AD     0.6346    0.6221    0.6283       614
NoDementia_Hy     0.9490    0.8638    0.9044      2217
       PD_LBD     0.5556    0.5932    0.5738        59
     Vascular     0.0788    0.5588    0.1382        34

     accuracy                         0.8040      2924
    macro avg     0.5545    0.6595    0.5612      2924
 weighted avg     0.8649    0.8040    0.8308      2924

Balanced accuracy: 0.659493397063192
[[ 382   91   10  131]
 [ 199 1915   17   86]
 [  13    6   35    5]
 [   8    6    1   19]]


HistGradientBoosting (tuned + weighted)
Balanced accuracy ≈ 0.613
Vascular recall ≈ 0.176
Worse than logistic regression.


In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np

w_train = y_train.map({
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 10.0,
    "Vascular": 20.0
}).astype(float).values

clf_hgb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.03,
        max_depth=8,
        max_iter=1200,
        min_samples_leaf=10,
        l2_regularization=0.0,
        random_state=42
    ))
])

clf_hgb.fit(X_train, y_train, model__sample_weight=w_train)
pred_hgb = clf_hgb.predict(X_test)

print("\n=== HistGradientBoosting tuned (weighted) ===")
print(classification_report(y_test, pred_hgb, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_hgb))
print(confusion_matrix(y_test, pred_hgb, labels=labels))


=== HistGradientBoosting tuned (weighted) ===
               precision    recall  f1-score   support

           AD     0.6676    0.7622    0.7118       614
NoDementia_Hy     0.9384    0.8868    0.9119      2217
       PD_LBD     0.5068    0.6271    0.5606        59
     Vascular     0.1091    0.1765    0.1348        34

     accuracy                         0.8471      2924
    macro avg     0.5555    0.6131    0.5798      2924
 weighted avg     0.8632    0.8471    0.8537      2924

Balanced accuracy: 0.6131470395701917
[[ 468  112   16   18]
 [ 202 1966   19   30]
 [  14    7   37    1]
 [  17   10    1    6]]


XGBoost (tuned + early stopping + strong class weighting)
Balanced accuracy ≈ 0.569
Vascular recall ≈ 0.029
Tree ensembles largely ignored vascular class despite weighting.


In [13]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]
label_to_int = {lab:i for i, lab in enumerate(labels)}
int_to_label = {i:lab for lab, i in label_to_int.items()}

y_train_int = y_train.map(label_to_int).astype(int)
y_test_int  = y_test.map(label_to_int).astype(int)

# stronger weights for tree models
TREE_WEIGHTS = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 10.0,
    "Vascular": 20.0
}
w_train = y_train.map(TREE_WEIGHTS).astype(float).values

# train/val split (no test leakage)
X_tr, X_val, y_tr_int, y_val_int, w_tr, w_val = train_test_split(
    X_train, y_train_int, w_train,
    test_size=0.2, random_state=42, stratify=y_train_int
)

# preprocess
Xtr_mat = preprocess.fit_transform(X_tr)
Xval_mat = preprocess.transform(X_val)
Xte_mat  = preprocess.transform(X_test)

# IMPORTANT: early_stopping_rounds goes in constructor now
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(labels),

    n_estimators=5000,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=1,
    subsample=0.85,
    colsample_bytree=0.85,

    reg_lambda=1.0,
    gamma=0.0,

    tree_method="hist",
    random_state=42,
    eval_metric="mlogloss",

    early_stopping_rounds=200   # <-- moved here
)

xgb.fit(
    Xtr_mat,
    y_tr_int,
    sample_weight=w_tr,
    eval_set=[(Xval_mat, y_val_int)],
    verbose=False
)

pred_int = xgb.predict(Xte_mat)
pred_xgb = pd.Series(pred_int).map(int_to_label)

print("\n=== XGBoost tuned (weighted + early stopping) ===")
print(classification_report(y_test, pred_xgb, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_xgb))
print(confusion_matrix(y_test, pred_xgb, labels=labels))


=== XGBoost tuned (weighted + early stopping) ===
               precision    recall  f1-score   support

           AD     0.6592    0.7720    0.7112       614
NoDementia_Hy     0.9342    0.8972    0.9153      2217
       PD_LBD     0.6415    0.5763    0.6071        59
     Vascular     0.0435    0.0294    0.0351        34

     accuracy                         0.8543      2924
    macro avg     0.5696    0.5687    0.5672      2924
 weighted avg     0.8602    0.8543    0.8560      2924

Balanced accuracy: 0.5687070609718583
[[ 474  119    8   13]
 [ 210 1989   10    8]
 [  18    6   34    1]
 [  17   15    1    1]]


Model B: Boosted trees + SHAP (XGBoost preferred, LightGBM fallback)

Try importing XGBoost / LightGBM

In [61]:
from sklearn.preprocessing import LabelEncoder

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

le = LabelEncoder()
le.fit(labels)

y_train_enc = le.transform(y_train)   # 0..3
y_test_enc  = le.transform(y_test)

print("Label mapping:", {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))})

Label mapping: {'AD': 0, 'NoDementia_Hy': 1, 'PD_LBD': 2, 'Vascular': 3}


In [62]:
use_model = None

try:
    from xgboost import XGBClassifier
    use_model = "xgb"
    print("Using XGBoost.")
except Exception as e:
    try:
        from lightgbm import LGBMClassifier
        use_model = "lgbm"
        print("Using LightGBM.")
    except Exception as e2:
        use_model = None
        print("Neither xgboost nor lightgbm found. Install one:")
        print("  pip install xgboost shap")
        print("or")
        print("  pip install lightgbm shap")


Using XGBoost.


Train boosted model (with sample weights)

In [63]:
print("y_train unique:", sorted(pd.Series(y_train).unique()))
print("y_train_enc unique:", sorted(np.unique(y_train_enc)))
print("counts:", pd.Series(y_train).value_counts())

y_train unique: ['AD', 'NoDementia_Hy', 'PD_LBD', 'Vascular']
y_train_enc unique: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
counts: label
NoDementia_Hy    8866
AD               2454
PD_LBD            238
Vascular          138
Name: count, dtype: int64


In [64]:
if use_model is not None:
    # sample weights based on your custom weights (strings)
    w_train = y_train.map(custom_w).astype(float).values

    if use_model == "xgb":
        booster = XGBClassifier(
            n_estimators=600,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="multi:softprob",
            num_class=len(labels),
            eval_metric="mlogloss",
            random_state=RANDOM_STATE
        )
    else:
        booster = LGBMClassifier(
            n_estimators=1200,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=0.1,
            objective="multiclass",
            random_state=RANDOM_STATE
        )

    clf_boost = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", booster)
    ])

    # IMPORTANT: fit with encoded labels
    clf_boost.fit(X_train, y_train_enc, model__sample_weight=w_train)

    # predict encoded and decode to string labels
    pred_boost_enc = clf_boost.predict(X_test)
    pred_boost = le.inverse_transform(pred_boost_enc.astype(int))

    print("=== Boosted Trees (custom sample_weight) ===")
    print(classification_report(y_test, pred_boost, digits=4))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_boost))
    print(confusion_matrix(y_test, pred_boost, labels=labels))

=== Boosted Trees (custom sample_weight) ===
               precision    recall  f1-score   support

           AD     0.6676    0.8046    0.7297       614
NoDementia_Hy     0.9378    0.8972    0.9170      2217
       PD_LBD     0.5000    0.4576    0.4779        59
     Vascular     0.2222    0.0588    0.0930        34

     accuracy                         0.8591      2924
    macro avg     0.5819    0.5545    0.5544      2924
 weighted avg     0.8639    0.8591    0.8592      2924

Balanced accuracy: 0.5545423076747463
[[ 494  105   12    3]
 [ 210 1989   14    4]
 [  20   12   27    0]
 [  16   15    1    2]]


In [65]:
# print("Boost model classes_:", getattr(clf_boost.named_steps["model"], "classes_", None))

Interpretability Part 1: Logistic Regression coefficient “top drivers”

In [78]:
def get_feature_names(preprocessor: ColumnTransformer):
    # numeric names
    num_names = preprocessor.named_transformers_["num"].get_feature_names_out(num_cols)
    # categorical names
    ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    cat_names = ohe.get_feature_names_out(cat_cols)
    return np.concatenate([num_names, cat_names])

# Fit already done; extract coefficients
pre = clf_lr.named_steps["preprocess"]
model = clf_lr.named_steps["model"]
feat_names = get_feature_names(pre)

coef = pd.DataFrame(model.coef_, columns=feat_names, index=model.classes_)
coef_path = os.path.join(OUT_DIR, "lr_coefficients_all_classes.csv")
coef.to_csv(coef_path)
print("Saved:", coef_path)

# Save top drivers per class
for cls in model.classes_:
    s = coef.loc[cls].sort_values(ascending=False)
    top_pos = s.head(15).reset_index()
    top_pos.columns = ["feature", "coef"]
    top_neg = s.tail(15).sort_values().reset_index()
    top_neg.columns = ["feature", "coef"]

    top_pos.to_csv(os.path.join(OUT_DIR, f"lr_top_positive_{cls}.csv"), index=False)
    top_neg.to_csv(os.path.join(OUT_DIR, f"lr_top_negative_{cls}.csv"), index=False)

print("Saved per-class top coefficient CSVs in output/.")


Saved: ..\output\lr_coefficients_all_classes.csv
Saved per-class top coefficient CSVs in output/.


Interpretability Part 2: SHAP for boosted trees (global + local)

In [79]:
import numpy as np
if not hasattr(np, "trapz") and hasattr(np, "trapezoid"):
    np.trapz = np.trapezoid
if not hasattr(np, "in1d") and hasattr(np, "isin"):
    np.in1d = np.isin


In [80]:
if use_model is not None:
    import shap
    import numpy as np
    import matplotlib.pyplot as plt

    # Transform X once (same features model sees)
    X_test_mat  = clf_boost.named_steps["preprocess"].transform(X_test)
    X_train_mat = clf_boost.named_steps["preprocess"].transform(X_train)

    feat_names_boost = get_feature_names(clf_boost.named_steps["preprocess"])

    # ---- SHAP (robust) ----
    # TreeExplainer fails for your XGBoost multiclass base_score vector, so use model-agnostic Explainer.
    # For speed, we use a small background sample from TRAIN.
    bg_n = min(300, X_train_mat.shape[0])
    rng = np.random.default_rng(RANDOM_STATE)
    bg_idx = rng.choice(X_train_mat.shape[0], size=bg_n, replace=False)
    background = X_train_mat[bg_idx]

    # For speed, also limit the explained test points (you can increase later)
    explain_n = min(600, X_test_mat.shape[0])
    X_explain = X_test_mat[:explain_n]

    # shap.Explainer expects a callable model; use predict_proba on the *pipeline*
    # but it expects input in original feature space (X_test), not preprocessed.
    # So we wrap a function that takes the preprocessed matrix and calls the model directly.
    base_model = clf_boost.named_steps["model"]

    # Need predict_proba on the underlying model using already-transformed matrices:
    def model_proba_on_matrix(M):
        return base_model.predict_proba(M)

    explainer = shap.Explainer(model_proba_on_matrix, background, feature_names=feat_names_boost)
    sv = explainer(X_explain)  # sv.values: (n_samples, n_features, n_classes)

    # --- Global summary plots for each class ---
    for i, cls in enumerate(labels):
        plt.figure()
        shap.summary_plot(
            sv.values[:, :, i],
            X_explain,
            feature_names=feat_names_boost,
            show=False,
            max_display=20
        )
        p = os.path.join(FIG_DIR, f"shap_summary_{cls}.png")
        plt.tight_layout()
        plt.savefig(p, dpi=200)
        plt.close()
        print("Saved:", p)

    # --- Local explanations: one example per predicted class ---
    # Get predicted classes for the explained subset
    pred_cls_enc = base_model.predict(X_explain)
    pred_cls = le.inverse_transform(pred_cls_enc.astype(int))

    for target in labels:
        idxs = np.where(pred_cls == target)[0]
        if len(idxs) == 0:
            continue
        j = int(idxs[0])

        # Waterfall for class index i
        i = labels.index(target)

        plt.figure()
        shap.plots.waterfall(
            shap.Explanation(
                values=sv.values[j, :, i],
                base_values=sv.base_values[j, i] if sv.base_values.ndim == 2 else sv.base_values[i],
                data=X_explain[j],
                feature_names=feat_names_boost
            ),
            max_display=15,
            show=False
        )
        p = os.path.join(FIG_DIR, f"shap_waterfall_pred_{target}.png")
        plt.tight_layout()
        plt.savefig(p, dpi=200)
        plt.close()
        print("Saved:", p)


PermutationExplainer explainer: 601it [01:02,  7.86it/s]                         
C:\Users\branjan\AppData\Local\Temp\ipykernel_35836\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figs_baseline\shap_summary_AD.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_35836\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figs_baseline\shap_summary_NoDementia_Hy.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_35836\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figs_baseline\shap_summary_PD_LBD.png


C:\Users\branjan\AppData\Local\Temp\ipykernel_35836\2746796708.py:39: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


Saved: ..\output\figs_baseline\shap_summary_Vascular.png
Saved: ..\output\figs_baseline\shap_waterfall_pred_AD.png
Saved: ..\output\figs_baseline\shap_waterfall_pred_NoDementia_Hy.png
Saved: ..\output\figs_baseline\shap_waterfall_pred_PD_LBD.png
Saved: ..\output\figs_baseline\shap_waterfall_pred_Vascular.png


In [81]:
# import numpy as np
# import inspect, os

# print("numpy version:", np.__version__)
# print("numpy file:", np.__file__)
# print("has trapz:", hasattr(np, "trapz"))
# print("cwd:", os.getcwd())


Subgroup performance (Sex, Race, Education bins)

In [82]:
def subgroup_report(model_pipe, X_test, y_test, group_series, group_name, le=None):
    rows = []
    for g in sorted(group_series.dropna().unique()):
        mask = (group_series == g)
        if mask.sum() < 30:
            continue

        yt = y_test[mask]
        yp = model_pipe.predict(X_test[mask])

        # If model outputs encoded numeric classes, decode to strings
        if le is not None:
            # yp could be numpy array of ints, or floats like 0.0, 1.0
            try:
                if np.issubdtype(np.array(yp).dtype, np.number):
                    yp = le.inverse_transform(np.array(yp).astype(int))
            except Exception:
                pass

        rows.append({
            "group_name": group_name,
            "group_value": str(g),
            "n": int(mask.sum()),
            "balanced_acc": balanced_accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
            "recall_AD": ( (yp == "AD") & (yt == "AD") ).sum() / max(1, (yt == "AD").sum()),
            "recall_NoDementia": ( (yp == "NoDementia_Hy") & (yt == "NoDementia_Hy") ).sum() / max(1, (yt == "NoDementia_Hy").sum()),
            "recall_PD_LBD": ( (yp == "PD_LBD") & (yt == "PD_LBD") ).sum() / max(1, (yt == "PD_LBD").sum()),
            "recall_Vascular": ( (yp == "Vascular") & (yt == "Vascular") ).sum() / max(1, (yt == "Vascular").sum()),
        })
    return pd.DataFrame(rows)


# Create subgroup variables from *raw X_test columns* (not encoded)
subgroups = {}

if "SEX" in X_test.columns:
    subgroups["SEX"] = X_test["SEX"]
if "RACE" in X_test.columns:
    # Optional: create White vs Non-White if you want
    subgroups["RACE"] = X_test["RACE"]
if "EDUC_BIN" in X_test.columns:
    subgroups["EDUC_BIN"] = X_test["EDUC_BIN"]

# Logistic regression subgroup table
tables = []
for name, s in subgroups.items():
    tables.append(subgroup_report(clf_lr, X_test, y_test, s, name))
sub_lr = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
p_lr = os.path.join(OUT_DIR, "subgroup_metrics_logreg.csv")
sub_lr.to_csv(p_lr, index=False)
print("Saved:", p_lr)
display(sub_lr.head(20))

# Boosted model subgroup table (if trained)
if use_model is not None:
    tables = []
    for name, s in subgroups.items():
        tables.append(subgroup_report(clf_boost, X_test, y_test, s, name, le=le))
    sub_boost = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    p_boost = os.path.join(OUT_DIR, "subgroup_metrics_boosted.csv")
    sub_boost.to_csv(p_boost, index=False)
    print("Saved:", p_boost)
    display(sub_boost.head(20))


Saved: ..\output\subgroup_metrics_logreg.csv


,group_name,group_value,n,balanced_acc,macro_f1,recall_AD,recall_NoDementia,recall_PD_LBD,recall_Vascular
0,SEX,1,1237,0.555541,0.558304,0.865385,1.0,0.239130,0.117647
1,SEX,2,1687,0.567293,0.577253,0.956954,1.0,0.076923,0.235294
2,RACE,1,2427,0.573130,0.579697,0.901923,1.0,0.218182,0.172414
3,RACE,2,367,0.566384,0.536648,0.932203,1.0,0.000000,0.333333
4,RACE,5,79,0.666667,0.650000,1.000000,1.0,0.000000,0.000000
5,EDUC_BIN,<=12,478,0.647905,0.673214,0.920290,1.0,0.307692,0.363636
6,EDUC_BIN,>12,2446,0.542108,0.544879,0.907563,1.0,0.173913,0.086957


Saved: ..\output\subgroup_metrics_boosted.csv


,group_name,group_value,n,balanced_acc,macro_f1,recall_AD,recall_NoDementia,recall_PD_LBD,recall_Vascular
0,SEX,1,1237,0.576997,0.585812,0.923077,1.0,0.326087,0.058824
1,SEX,2,1687,0.496689,0.486508,0.986755,1.0,0.000000,0.000000
2,RACE,1,2427,0.563822,0.571135,0.948077,1.0,0.272727,0.034483
3,RACE,2,367,0.495763,0.483871,0.983051,1.0,0.000000,0.000000
4,RACE,5,79,0.666667,0.650000,1.000000,1.0,0.000000,0.000000
5,EDUC_BIN,<=12,478,0.529404,0.527740,0.963768,1.0,0.153846,0.000000
6,EDUC_BIN,>12,2446,0.569442,0.579151,0.951681,1.0,0.282609,0.043478


In [83]:
def subgroup_class_counts(y, group):
    t = pd.crosstab(group, y)
    return t

for name, s in subgroups.items():
    print("\n==", name, "==")
    display(subgroup_class_counts(y_test, s))



== SEX ==


label,AD,NoDementia_Hy,PD_LBD,Vascular
SEX,,,,
1,312,862,46,17
2,302,1355,13,17



== RACE ==


label,AD,NoDementia_Hy,PD_LBD,Vascular
RACE,,,,
1,520,1823,55,29
2,59,301,4,3
3,1,14,0,0
4,1,2,0,0
5,19,58,0,2
50,14,14,0,0



== EDUC_BIN ==


label,AD,NoDementia_Hy,PD_LBD,Vascular
EDUC_BIN,,,,
<=12,138,316,13,11
>12,476,1901,46,23


Ablation study (prove the model improves as you add real clinical signal)

In [15]:
# Define feature groups (adjust if needed based on columns you actually have)
DEMOG = [c for c in ["NACCAGE","SEX","RACE","HISPANIC","EDUC","NACCUDSD"] if c in X.columns]
MED = [c for c in [
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA","DEP2YRS",
    "BPSYS","BPDIAS","CVCHF","CVANGINA","CVHATT","PREVSTK","STROKCOG","CVDCOG","STKIMAG",
    "CVDIMAG","CVDIMAG1","CVDIMAG2","CVDIMAG3","CVDIMAG4"
] if c in X.columns]
COG_FUNC = [c for c in ["NACCMMSE","NACCMOCA","TRAILA","TRAILB","ANIMALS","BOSTON","CDRSUM","CDRGLOB"] if c in X.columns]

ABLATIONS = {
    "Demographics": DEMOG,
    "Demog+Medical": DEMOG + MED,
    "Demog+Medical+Cognition": DEMOG + MED + COG_FUNC,
    "Full": list(X.columns)
}

def train_eval_lr_on_cols(cols):
    Xtr = X_train[cols].copy()
    Xte = X_test[cols].copy()

    # Ensure categorical code vars are OBJECT (not pandas "string") and missing is np.nan
    cat_codes = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD", "EDUC_BIN"] if c in cols and c in Xtr.columns]
    for c in cat_codes:
        # Convert values to object strings, preserve missing as np.nan
        Xtr[c] = Xtr[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)
        Xte[c] = Xte[c].map(lambda v: np.nan if pd.isna(v) else str(v)).astype(object)

    # Also convert any pd.NA left in the frame to np.nan
    Xtr = Xtr.replace({pd.NA: np.nan})
    Xte = Xte.replace({pd.NA: np.nan})

    cat_cols_ = [c for c in Xtr.columns if Xtr[c].dtype == "object"]
    num_cols_ = [c for c in Xtr.columns if c not in cat_cols_]

    pre_ = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), num_cols_),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols_),
        ],
        remainder="drop"
    )

    clf = Pipeline(steps=[
        ("preprocess", pre_),
        ("model", LogisticRegression(max_iter=8000, class_weight=custom_w, C=0.1))
    ])

    clf.fit(Xtr, y_train)
    yp = clf.predict(Xte)

    return {
        "set": "",
        "n_features": len(cols),
        "macro_f1": f1_score(y_test, yp, average="macro"),
        "balanced_acc": balanced_accuracy_score(y_test, yp)
    }


rows = []
for name, cols in ABLATIONS.items():
    if len(cols) == 0:
        continue
    r = train_eval_lr_on_cols(cols)
    r["set"] = name
    rows.append(r)

ab = pd.DataFrame(rows).sort_values("n_features")
p_ab = os.path.join(OUT_DIR, "ablation_logreg.csv")
ab.to_csv(p_ab, index=False)
print("Saved:", p_ab)
display(ab)


Saved: ..\output\ablation_logreg.csv


,set,n_features,macro_f1,balanced_acc
0,Demographics,6,0.419786,0.432871
1,Demog+Medical,6,0.419786,0.432871
2,Demog+Medical+Cognition,14,0.413076,0.431179
3,Full,37,0.615098,0.630656


## Export run summary (for comparing approaches)
This cell writes a compact CSV with the key metrics from this baseline notebook so you can compare against the hierarchical notebook.


In [16]:
import pandas as pd
import os
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score

# Determine which baseline model(s) exist in this notebook
# - Logistic regression: clf_lr
# - Boosted (LightGBM/XGBoost): clf_boost (optional)
rows = []

def _add_row(name, y_true, y_pred):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    rows.append({
        "approach": "baseline",
        "model": name,
        "n_test": int(len(y_true)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "vas_precision": float(rep.get("Vascular", {}).get("precision", 0.0)),
        "vas_recall": float(rep.get("Vascular", {}).get("recall", 0.0)),
        "vas_f1": float(rep.get("Vascular", {}).get("f1-score", 0.0)),
        "ad_recall": float(rep.get("AD", {}).get("recall", 0.0)),
        "pd_recall": float(rep.get("PD_LBD", {}).get("recall", 0.0)),
        "nodem_recall": float(rep.get("NoDementia_Hy", {}).get("recall", 0.0)),
    })

# Logistic regression predictions should already exist as y_pred_lr; if not, compute
try:
    y_pred_lr
except NameError:
    y_pred_lr = clf_lr.predict(X_test)

_add_row("logreg", y_test, y_pred_lr)

# Boost model (optional)
if "clf_boost" in globals():
    try:
        y_pred_boost
    except NameError:
        y_pred_boost = clf_boost.predict(X_test)
        # If label encoder exists, decode numeric predictions
        if "le" in globals() and le is not None:
            try:
                import numpy as np
                if np.issubdtype(np.array(y_pred_boost).dtype, np.number):
                    y_pred_boost = le.inverse_transform(np.array(y_pred_boost).astype(int))
            except Exception:
                pass
    _add_row("boosted", y_test, y_pred_boost)

summary_df = pd.DataFrame(rows)

# Attach pointers to the detailed artifacts this notebook already writes
artifact_paths = {
    "subgroup_logreg": os.path.join(OUT_DIR, "subgroup_metrics_logreg.csv"),
    "subgroup_boosted": os.path.join(OUT_DIR, "subgroup_metrics_boosted.csv"),
    "ablation_logreg": os.path.join(OUT_DIR, "ablation_logreg.csv"),
}

for k,v in artifact_paths.items():
    summary_df[k] = v if os.path.exists(v) else ""

p_sum = os.path.join(OUT_DIR, "run_summary_baseline.csv")
summary_df.to_csv(p_sum, index=False)
print("Saved:", p_sum)
display(summary_df)


Saved: ..\output\run_summary_baseline.csv


,approach,model,n_test,balanced_acc,macro_f1,vas_precision,vas_recall,vas_f1,ad_recall,pd_recall,nodem_recall,subgroup_logreg,subgroup_boosted,ablation_logreg
0,baseline,logreg,2924,0.631063,0.615385,0.380952,0.235294,0.290909,0.799674,0.610169,0.879116,..\output\subgroup_metrics_logreg.csv,..\output\subgroup_metrics_boosted.csv,..\output\ablation_logreg.csv
